# Notebook 02: SDAE Dimensionality Reduction

The pilot plant has 95 process variables (89 numeric + 6 one-hot sampling point).
We compress these to 16 latent features using a Stacked Denoising Autoencoder (SDAE)
before feeding into GRU/Transformer, following Chai et al. (2026).

## Architecture
```
Input (95) + AWGN noise
  -> Linear(95, 64) -> ReLU -> Dropout(0.1)
  -> Linear(64, 32) -> ReLU -> Dropout(0.1)
  -> Linear(32, 16)  [LATENT]
  -> Linear(16, 32) -> ReLU
  -> Linear(32, 64) -> ReLU
  -> Linear(64, 95)  [RECONSTRUCTION]
```
Compression ratio: 95 → 16 (16.8% of original dimensionality).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from src import config as cfg

# Load SDAE training history
hist = np.load(cfg.OUT_METRICS / 'history_sdae.npy', allow_pickle=True).item()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist['train_loss'], label='Train reconstruction MSE', color='steelblue')
ax.plot(hist['val_loss'],   label='Val reconstruction MSE',   color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('SDAE Pre-training Loss Curves')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Best val MSE: {min(hist['val_loss']):.5f}")
print(f"Note: higher val loss reflects distribution shift between train/val runs (known limitation)")


In [ ]:
from IPython.display import Image
Image(str(cfg.OUT_FIGURES / '03_sdae_latent_space.png'))


## Latent space structure
The 2D PCA projection of encoded val-run features shows temporal structure
(colour = time step), confirming the SDAE captures dynamic process evolution
rather than static snapshots.

In [ ]:
# Show reconstruction quality on a sample from val run
import torch
from src.data_loading import load_all_runs
from src.autoencoder import SDAE, encode_dataset
from src.windowing import _get_feature_matrix
from src.utils import load_scaler
from sklearn.preprocessing import StandardScaler

runs = load_all_runs(['140313_1'])  # val run
feat_scaler = load_scaler(cfg.OUT_METRICS / 'feature_scaler.pkl')
feats_raw = _get_feature_matrix(runs['140313_1']['df'], use_label_onehot=True)
feats_sc  = feat_scaler.transform(feats_raw)

sdae = SDAE(feats_sc.shape[1])
sdae.load_state_dict(torch.load(cfg.OUT_CHECKPOINTS / 'sdae.pt', weights_only=True))
sdae.eval()

with torch.no_grad():
    x_tensor = torch.from_numpy(feats_sc).float()
    x_hat, z = sdae(x_tensor)

recon_err = ((x_tensor - x_hat)**2).mean(dim=1).numpy()
print(f"Latent dim: {z.shape[1]}")
print(f"Val run reconstruction MSE per timestep: mean={recon_err.mean():.4f} std={recon_err.std():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(recon_err, color='steelblue', lw=1)
axes[0].set_title('Per-timestep reconstruction error (val run)'); axes[0].set_xlabel('Time step')
axes[1].hist(recon_err, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Reconstruction error distribution'); axes[1].set_xlabel('MSE')
plt.tight_layout(); plt.show()
